## Setup


In [ ]:
from dotenv import load_dotenv

# set any relevant environment variables, such as HF_TOKEN,
# which speed up downloads of models
load_dotenv()

## Freeing memory

In [ ]:
import gc

import GPUtil
import torch

gc.collect()
torch.cuda.empty_cache()
GPUtil.showUtilization()

## Tokenization with BERT


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
input_text = "What is unhappiness?"
tokens = tokenizer.tokenize(input_text, return_tensors="pt")

print(f"{tokens = }")

## Contextual token embeddings with BERT


In [ ]:
import torch
from transformers import BertModel, BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

input_text = """
After a long day at work, Sarah decided to relax by taking her 
dog for a walk in the park. As they strolled along the 
tree-lined paths, Sarah's dog, Max, eagerly sniffed around, 
chasing after squirrels and birds. Sarah smiled as she watched 
Max enjoy himself, feeling grateful for the companionship and 
joy that her furry friend brought into her life."""

tokens = tokenizer(input_text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**tokens)

last_hidden_states = outputs.last_hidden_state
print("Token embeddings:")
# strict=True added by lint fix (B905): check input lengths match when running
for token, embedding in zip(tokens["input_ids"][0], last_hidden_states[0], strict=True):
    word = tokenizer.decode(int(token))
    print(f"{word}: {embedding}")

## Visualizing token embeddings with t-SNE


In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=5, random_state=42)
embeddings_tsne = tsne.fit_transform(last_hidden_states[0])

plt.figure(figsize=(10, 8))
plt.scatter(embeddings_tsne[:, 0], embeddings_tsne[:, 1], marker="o")
for i, word in enumerate(tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])):
    plt.annotate(word, xy=(embeddings_tsne[i, 0], embeddings_tsne[i, 1]), fontsize=10)
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.title("t-SNE Visualization of Token Embeddings")
plt.show()

## Positional embeddings


In [ ]:
import torch
from transformers import BertModel, BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

input_text = """
After a long day at work, Sarah decided to relax by taking her 
dog for a walk in the park. As they strolled along the 
tree-lined paths, Sarah's dog, Max, eagerly sniffed around, 
chasing after squirrels and birds. Sarah smiled as she watched 
Max enjoy himself, feeling grateful for the companionship and 
joy that her furry friend brought into her life."""

tokens = tokenizer(input_text, return_tensors="pt")
embeddings = model.embeddings  # A
positional_embeddings = embeddings.position_embeddings.weight
position_ids = torch.arange(tokens["input_ids"].size(1), dtype=torch.long).unsqueeze(0)
input_positional_embeddings = positional_embeddings[position_ids]

print("Positional embeddings shape:", input_positional_embeddings.shape)
print("Positional embeddings for each token:")

# strict=True added by lint fix (B905): check input lengths match when running
for token_id, pos_embedding in zip(
    tokens["input_ids"][0], input_positional_embeddings[0], strict=True
):
    token = tokenizer.decode([token_id])
    print(f"{token}: {pos_embedding}")

## Sentiment analysis with DistilBERT (manual tokenizer and model)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

In [ ]:
import torch

text = "I loved the movie, it was fantastic!"

inputs = tokenizer(text, return_tensors="pt")
print(inputs)

In [ ]:
outputs = model(**inputs)
print(outputs)

In [ ]:
predicted_label = torch.argmax(outputs.logits)
sentiment = "positive" if predicted_label == 1 else "negative"
print("Predicted sentiment:", sentiment)

## Text classification with pipelines


In [ ]:
from transformers import pipeline

try:
    dummy_pipeline = pipeline(task="dummy")
except Exception as e:
    print(e)

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)

In [ ]:
pipe = pipeline("text-classification")

In [ ]:
from transformers import pipeline

classifier = pipeline(
    task="text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)

In [ ]:
classifier = pipeline(
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

In [ ]:
classifier = pipeline(
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", device="cuda"
)

In [ ]:
review1 = """From the warm welcome to the exquisite dishes and impeccable
 service, dining at Gourmet Haven is an unforgettable experience that
 leaves you eager to return."""

review2 = """Despite high expectations, our experience at Savor Bistro 
 fell short; the food was bland, service was slow, and the overall
 atmosphere lacked charm, leaving us disappointed and unlikely to
 revisit."""

In [ ]:
print(classifier(review1))

In [ ]:
print(classifier([review1, review2]))

## Question detection


In [ ]:
from transformers import pipeline

question_classifier = pipeline(
    "text-classification", model="huaen/question_detection", from_pt=True
)

In [ ]:
response = question_classifier(
    """Have you ever pondered the mysteries that lie beneath 
    the surface of everyday life?"""
)
print(response)

In [ ]:
response = question_classifier(
    """"Life is a journey that must be traveled, no matter 
    how bad the roads and accommodations." - Oliver Goldsmith"""
)
print(response)

## Language detection


In [ ]:
language_classifier = pipeline(
    "text-classification", model="papluca/xlm-roberta-base-language-detection"
)

response = language_classifier("日本の桜は美しいです。")
print(response)

## Spam detection


In [ ]:
spam_classifier = pipeline(
    "text-classification",
    model="Delphia/twitter-spam-classifier",
)

response = spam_classifier(
    """Congratulations! You've been selected as the winner of our 
    exclusive prize draw. Claim your reward now by clicking on 
    the link below!"""
)

print(response)

In [ ]:
response = spam_classifier(
    """Hi Jimmy, I hope you're doing well. I just wanted to remind 
    you about our meeting tomorrow at 10 AM in conference room A. 
    Please let me know if you have any questions or need any 
    further information. Looking forward to seeing you there!"""
)

print(response)

## Text generation with GPT-2


In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="openai-community/gpt2")
generator("In this course, we will teach you how to")

In [ ]:
generator(
    "In this course, we will teach you how to", max_length=50, num_return_sequences=3
)

## Summarization with Qwen


In [ ]:
# Summarization and translation are deprecated pipeline tasks in transformers v5.
# They will be transformed into text-generation, where the model is an LLM prompted
# with the task instruction. See https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#text-pipelines-that-should-just-be-llms
from transformers import pipeline

summarizer = pipeline(
    "text-generation", model="Qwen/Qwen3-4B-Instruct-2507", device="cpu"
)
# summarization task removed  https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#pipelines
# set device cpu for out of memory

In [ ]:
article = """ 
A quantum computer is a computer that exploits quantum mechanical 
phenomena. At small scales, physical matter exhibits properties of 
both particles and waves, and quantum computing leverages this 
behavior using specialized hardware. Classical physics cannot 
explain the operation of these quantum devices, and a scalable 
quantum computer could perform some calculations exponentially 
faster than any modern "classical" computer. In particular, a 
large-scale quantum computer could break widely used encryption 
schemes and aid physicists in performing physical simulations; 
however, the current state of the art is still largely 
experimental and impractical.

The basic unit of information in quantum computing is the qubit, 
similar to the bit in traditional digital electronics. Unlike a 
classical bit, a qubit can exist in a superposition of its two 
"basis" states, which loosely means that it is in both states 
simultaneously. When measuring a qubit, the result is a 
probabilistic output of a classical bit. If a quantum computer 
manipulates the qubit in a particular way, wave interference 
effects can amplify the desired measurement results. The design 
of quantum algorithms involves creating procedures that allow a 
quantum computer to perform calculations efficiently.

Physically engineering high-quality qubits has proven challenging. 
If a physical qubit is not sufficiently isolated from its 
environment, it suffers from quantum decoherence, introducing noise 
into calculations. National governments have invested heavily in 
experimental research that aims to develop scalable qubits with 
longer coherence times and lower error rates. Two of the most 
promising technologies are superconductors (which isolate an 
electrical current by eliminating electrical resistance) and ion 
traps (which confine a single atomic particle using electromagnetic 
fields).

Any computational problem that can be solved by a classical computer 
can also be solved by a quantum computer.[2] Conversely, any problem 
that can be solved by a quantum computer can also be solved by a 
classical computer, at least in principle given enough time. In other 
words, quantum computers obey the Church–Turing thesis. This means 
that while quantum computers provide no additional advantages over 
classical computers in terms of computability, quantum algorithms 
for certain problems have significantly lower time complexities than 
corresponding known classical algorithms. Notably, quantum computers 
are believed to be able to solve certain problems quickly that no 
classical computer could solve in any feasible amount of time—a feat 
known as "quantum supremacy." The study of the computational 
complexity of problems with respect to quantum computers is known as 
quantum complexity theory.
"""

message = [{"role": "user", "content": f"Summarize the following text:\n\n{article}"}]

summarised = summarizer(message, min_length=100, max_new_tokens=250, do_sample=False)
# extractive summarization
print(summarised[0]["generated_text"][-1]["content"])

In [ ]:
print(len(article), len(summarised[0]["generated_text"][-1]["content"]))

In [ ]:
summarised_abs = summarizer(
    message, min_length=100, max_new_tokens=250, do_sample=False
)
print(
    print(summarised_abs[0]["generated_text"][-1]["content"])
)  # abstractive summarization

## Translation with Qwen


In [ ]:
from transformers import pipeline

# model="google-t5/t5-base"
model = "Qwen/Qwen3-1.7B"  # "Qwen/Qwen3-4B-Instruct-2507"
device = "cuda"
text_generator = pipeline(
    "text-generation",
    model=model,
    device=device,
    dtype=torch.bfloat16,
    tokenizer_encode_kwargs={"enable_thinking": False},
)

In [ ]:
text = "How are you?"
# google-t5/t5-base defaulted to translating into german
message = [
    {
        "role": "user",
        "content": rf"Translate the following text into German:\n\n{text}",
    }
]
response = text_generator(message)
print(response[0]["generated_text"][-1]["content"])

## Translation to French and German


In [ ]:
sentence = (
    "Wikipedia is hosted by the Wikimedia Foundation, a non-profit organization "
    "that also hosts a range of other projects."
)

for language in ["French", "German"]:
    message = [
        {
            "role": "user",
            "content": f"Translate the following text into {language}:\n\n{sentence}",
        }
    ]
    response = text_generator(message)
    print(response[0]["generated_text"][-1]["content"])

>La Wikipedia est hébergée par le Fonds Wikimedia, une organisation non profit qui également héberge des autres projets.
>Wikipedia ist auf dem Wikimедiat Foundation gelegt, einer nicht-großen Organisation, die auch andere Projekte hosting.

Qwen/Qwen2.5-0.5B-Instruct makes mistakes in the german (wikimediat, not big, rather than non-profit, hosting verb?)

## Translation to Chinese and back


In [ ]:
sentence_en = (
    "Wikipedia is hosted by the Wikimedia Foundation, a non-profit organization "
    "that also hosts a range of other projects."
)
message = [
    {
        "role": "user",
        "content": f"Translate the following text into Chinese:\n\n{sentence_en}",
    }
]
print(text_generator(message)[0]["generated_text"][-1]["content"])

In [ ]:
sentence_zh = "维基百科是维基百科基金会主办的,是一家非营利组织,还主办了许多其他项目。"
message = [
    {
        "role": "user",
        "content": f"Translate the following text into English:\n\n{sentence_zh}",
    }
]
print(text_generator(message)[0]["generated_text"][-1]["content"])

## Zero-shot text classification


In [ ]:
from transformers import pipeline

zero_shot_classifier = pipeline(
    "zero-shot-classification", model="joeddav/xlm-roberta-large-xnli"
)

In [ ]:
text1 = """
"In the intricate realm of global affairs, the interplay of power, 
diplomacy, and governance stands as a defining force in the 
trajectory of nations. Amidst fervent debates in legislative 
chambers and pivotal dialogues among world leaders, ideologies 
clash and policies take shape, shaping the course of societies. 
Issues such as economic disparity, environmental stewardship, and 
human rights take precedence, driving conversations and shaping 
public sentiment. In an age of digital interconnectedness, social 
media platforms have emerged as influential channels for discourse 
and activism, amplifying voices and reshaping narratives with 
remarkable speed and breadth. As citizens grapple with the 
complexities of contemporary governance, the pursuit of accountable 
and transparent leadership remains paramount, reflecting an 
enduring quest for fairness and inclusivity in societal governance."
"""

text2 = """
In the tender tapestry of human connection, romance weaves its 
delicate threads, binding hearts in a dance of passion and longing. 
From the flutter of a first glance to the warmth of an intimate 
embrace, love blooms in the most unexpected places, transcending 
barriers of time and circumstance. In the gentle caress of a hand 
and the whispered promises of affection, two souls find solace in 
each other's embrace, navigating the complexities of intimacy with 
tender care. As the sun sets and stars illuminate the night sky, 
lovers share stolen moments of intimacy, lost in the intoxicating 
rhythm of each other's presence. In the symphony of love, every 
glance, every touch, speaks volumes of a shared bond that defies 
explanation, leaving hearts entwined in an eternal embrace.
"""

In [ ]:
candidate_labels = ["technology", "politics", "business", "romance"]
prediction = zero_shot_classifier(text1, candidate_labels, multi_label=True)

In [ ]:
import pandas as pd

display(pd.DataFrame(prediction).drop(["sequence"], axis=1))

In [ ]:
prediction = zero_shot_classifier([text1, text2], candidate_labels, multi_label=True)
display(pd.DataFrame(prediction).drop(["sequence"], axis=1))

## Zero-shot image classification


In [ ]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-image-classification", model="openai/clip-vit-large-patch14-336"
)

In [ ]:
labels_for_classification = ["airplane", "car", "train"]
scores = classifier(
    "Emirates_Airbus_A380-861_A6-EER_MUC_2015_04.jpg",
    candidate_labels=labels_for_classification,
)
pd.DataFrame(scores)

## Question answering


In [ ]:
text = """
The English name of "Singapore" is an anglicisation of the native 
Malay name for the country, Singapura (pronounced [siŋapura]), 
which was in turn derived from the Sanskrit word for 'lion city' 
(Sanskrit: सिंहपुर; romanised: Siṃhapura; Brahmi: 𑀲𑀺𑀁𑀳𑀧𑀼𑀭; literally 
"lion city"; siṃha means 'lion', pura means 'city' or 'fortress' ).
Pulau Ujong was one of the earliest references to Singapore Island, 
which corresponds to a Chinese account from the third century 
referred to a place as Pú Luó Zhōng (Chinese: 蒲 羅 中), a 
transcription of the Malay name for 'island at the end of a 
peninsula'. Early references to the name Temasek (or Tumasik) are 
found in the Nagarakretagama, a Javanese eulogy written in 1365, 
and a Vietnamese source from the same time period. The name possibly 
means Sea Town, being derived from the Malay tasek, meaning 'sea' or 
'lake'. The Chinese traveller Wang Dayuan visited a place around 1330 
named Danmaxi (Chinese: 淡馬錫; pinyin: Dànmǎxí; Wade–Giles: Tan Ma Hsi) 
or Tam ma siak, depending on pronunciation; this may be a transcription 
of Temasek, alternatively, it may be a combination of the Malay Tanah
meaning 'land' and Chinese xi meaning 'tin', which was traded on the 
island
"""

In [ ]:
question = "What is the meaning of Singapura?"
message = [
    {
        "role": "user",
        "content": (
            "Answer the question using only the context below.\n\n"
            f"Context:\n{text}\n\nQuestion: {question}"
        ),
    }
]
response = text_generator(message)
print(response[0]["generated_text"][-1]["content"])